In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import LinearRegression
from sklearn.compose import TransformedTargetRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.cluster import KMeans
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import cross_val_score, GridSearchCV, RandomizedSearchCV

### Downloading data

In [3]:
from pathlib import Path
import tarfile
import urllib.request


def loading_house_data():
    
    zipFile_path = Path("data/housing.tgz")

    if not zipFile_path.is_file():
        Path("data").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, zipFile_path)
    
    tarfile.open(zipFile_path).extractall(path="data")

    return pd.read_csv(Path("data/housing/housing.csv"))

In [4]:
housing = loading_house_data()

### Information and studying data

In [5]:
housing
# housing.info()
housing['ocean_proximity'].value_counts()
# housing.describe() # Statistically helpful feature


# Visualization using matplotlib
# housing.hist(bins=50, figsize=(12,8))
# plt.show()

ocean_proximity
<1H OCEAN     9136
INLAND        6551
NEAR OCEAN    2658
NEAR BAY      2290
ISLAND           5
Name: count, dtype: int64

### Adding id's for identification

#### id'ing via indexing

In [6]:
# housing_with_id = housing.reset_index()
# housing_with_id.head()

#### id'ing via existing features

In [7]:
# housing_with_id2 = housing['longitude'] * 1000 + housing['latitude']
# housing_with_id2.describe()

### Creating test and training set

#### 1. Hashing

In [8]:
from zlib import crc32

# Function to decide test set using hashing

def is_id_in_test_set(identifier, test_ratio):
    return crc32(np.int64(identifier)) < test_ratio * 2 ** 32

def split_data(data, test_ratio, id_column):
    ids = data[id_column]
    in_test_set = ids.apply(lambda id_: is_id_in_test_set(id_, test_ratio))

    return data.loc[~in_test_set], data.loc[in_test_set]

In [9]:
# train_set, test_set = split_data(housing_with_id, 0.2, "index") 

#### 2. Using sklearn

In [10]:
# train_set, test_set = train_test_split(housing, test_size=0.2, random_state=42)

### Stratifying the median_income category

In [11]:
housing["income_cat"] = pd.cut(housing["median_income"], bins= [0., 1.5, 3.0, 4.5, 6.0, np.inf], labels= ["less than 15k", "less than 30k", "less than 45k", "less than 60k", "above 60k"])

In [12]:
# housing["income_cat"].value_counts().sort_index().plot.bar(rot=0, grid=True)
# plt.xlabel("Income category")
# plt.ylabel("Number of districts")
# plt.show()

In [13]:
# Splitting of test and training data post stratification

strat_train_set, strat_test_set = train_test_split(housing, test_size=0.2, stratify= housing["income_cat"], random_state=42)

# It's an accepted practice to drop the newly formed income category after stratification is done

for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

housing_train = strat_train_set.copy()

In [14]:
# housing_train.plot(kind="scatter", x='longitude', y='latitude', grid=True, s=housing_train["population"]/100, label="population", c="median_house_value", colorbar=True, legend=True, figsize=(10,7))

# plt.show()

In [15]:
# corr_matrix = housing_train.corr(numeric_only=True)

In [16]:
# corr_matrix["median_house_value"].sort_values(ascending= False)

By using the correlation matrix, we can find out different correlations between median house value and other attributes. The top attributes are analyzed inorder to find reasonable relations to use in designing the model.

In [17]:
# from pandas.plotting import scatter_matrix

# attributes = ["median_house_value", "median_income", "total_rooms", "housing_median_age"]

# scatter_matrix(housing_train[attributes], figsize=(12,8))

The above scatter plots indicate a strong relation between median house value and median income which can become the starting point of the project and model

In [18]:
# housing_train.plot(kind="scatter", y="median_house_value", x= "median_income", alpha=0.3, grid= True)

### Feature engineered that have better correlation with median house value

In [19]:
housing_train["population per household*"] = housing_train["population"] / housing_train["households"]
housing_train["Rooms per household*"] = housing_train["total_rooms"] / housing_train["households"]
housing_train["bedrooms to rooms ratio*"] = housing_train["total_bedrooms"] / housing_train["total_rooms"]

In [20]:
# corr_matrix = housing_train.corr(numeric_only=True)

In [21]:
# corr_matrix["median_house_value"].sort_values(ascending= False)

# Preparing data for cleaning and use

In [22]:
# Creating a fresh copy to work with

housing_features = strat_train_set.drop("median_house_value", axis=1) # drops median house values as labels seperately
housing_labels = strat_train_set["median_house_value"].copy() # Will use the labels as target values later


### Dealing with missing data and encoding categorical data

## Feature Scaling

In [23]:
# Cluster Similarity class
class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y=None, sample_weight=None):
        self.kmeans_ = KMeans(self.n_clusters, random_state=self.random_state)
        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self  
    
    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)
    
    def get_feature_names_out(self, names=None):
        return [f"Cluster {i} similarity" for i in range(self.n_clusters)]

## Pipeline

In [32]:
def column_ratio(X):
    return X[:,[0]] / X[:, [1]]

def ratio_name(function_transformer, feature_name_in):
    return ["ratio"]

def ratio_pipeline():
    return make_pipeline(SimpleImputer(strategy="median"), FunctionTransformer(column_ratio, feature_names_out=ratio_name), StandardScaler())

def log_pipeline():
    return make_pipeline(SimpleImputer(strategy="median"), FunctionTransformer(np.log, feature_names_out="one-to-one"), StandardScaler())

basic_cat_pipeline = make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore", sparse_output=False))

cluster_similarity_geo = ClusterSimilarity(n_clusters=10, gamma=1., random_state=42)

basic_num_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())

preprocessing = ColumnTransformer([
    ("bedrooms", ratio_pipeline(), ["total_bedrooms", "total_rooms"]), 
    ("rooms_per_house", ratio_pipeline(), ["total_rooms", "households"]),
    ("people_per_house", ratio_pipeline(), ["population", "households"]), 
    ("log", log_pipeline(), ["total_bedrooms", "total_rooms", "population",
    "households", "median_income"]), 
    ("geo", cluster_similarity_geo, ["latitude", "longitude"]), 
    ("cat", basic_cat_pipeline, ["ocean_proximity"])
], remainder= basic_num_pipeline)

housing_prepared = preprocessing.fit_transform(housing_features)

housing_prepared_df = pd.DataFrame(housing_prepared, columns=preprocessing.get_feature_names_out(), index=housing_features.index)

housing_prepared_df


,bedrooms__ratio,rooms_per_house__ratio,people_per_house__ratio,log__total_bedrooms,log__total_rooms,log__population,log__households,log__median_income,geo__Cluster 0 similarity,geo__Cluster 1 similarity,...,geo__Cluster 6 similarity,geo__Cluster 7 similarity,geo__Cluster 8 similarity,geo__Cluster 9 similarity,cat__ocean_proximity_<1H OCEAN,cat__ocean_proximity_INLAND,cat__ocean_proximity_ISLAND,cat__ocean_proximity_NEAR BAY,cat__ocean_proximity_NEAR OCEAN,remainder__housing_median_age
2231,-0.462287,0.202583,-0.065177,-0.267471,-0.104954,-0.534868,-0.286777,-0.341320,1.018578e-01,2.179218e-10,...,3.322838e-01,7.658401e-05,8.618445e-03,2.120741e-13,1.0,0.0,0.0,0.0,0.0,-0.529716
3397,-1.019957,0.756987,-0.037934,-0.512760,-0.053237,-0.602807,-0.518793,0.938396,2.303377e-01,1.813241e-14,...,9.937217e-01,6.107816e-03,2.722337e-04,8.364526e-18,0.0,0.0,0.0,1.0,0.0,1.847981
9533,-1.230785,0.821066,0.000559,0.825873,1.372854,1.019070,0.917273,1.824343,1.170276e-15,6.793116e-01,...,4.492003e-18,3.863186e-26,2.071098e-06,7.727507e-01,1.0,0.0,0.0,0.0,0.0,-1.084512
3482,0.012760,-0.257853,0.084810,-0.731764,-0.760340,-0.194355,-0.668155,-1.225011,2.183606e-15,4.954974e-01,...,2.916324e-18,1.473988e-25,3.332841e-06,9.982419e-01,0.0,1.0,0.0,0.0,0.0,0.896902
16621,0.220219,-0.513460,-0.116925,-2.508162,-2.563333,-2.928141,-2.327277,0.476456,1.715814e-12,9.735584e-01,...,2.353749e-14,4.312835e-22,1.308346e-04,3.512186e-01,1.0,0.0,0.0,0.0,0.0,0.817645
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12443,-0.452217,0.002398,0.008717,-0.834873,-0.659797,-0.580399,-0.737493,0.145235,1.413744e-12,9.873261e-01,...,1.710615e-14,3.450146e-22,1.208199e-04,3.882213e-01,1.0,0.0,0.0,0.0,0.0,1.134672
1955,2.377845,-1.231488,0.043303,-1.942018,-2.649975,-1.335719,-1.655405,-2.115003,5.743728e-13,9.984426e-01,...,5.514482e-15,1.097940e-22,7.532254e-05,4.741986e-01,1.0,0.0,0.0,0.0,0.0,1.530955
3551,-0.322160,-0.440661,0.143538,-1.464521,-1.329478,-0.436420,-1.117318,0.164184,1.055648e-12,9.966911e-01,...,8.743979e-15,2.648438e-22,1.113081e-04,4.989688e-01,1.0,0.0,0.0,0.0,0.0,0.976159
11607,0.151620,-0.179771,-0.054281,0.393344,0.276072,0.155478,0.342791,0.105309,2.665515e-13,9.722628e-01,...,1.675944e-15,4.545358e-23,5.221433e-05,6.188294e-01,1.0,0.0,0.0,0.0,0.0,1.451698


## Cross Validation and model selection

### Linear Regression

In [25]:
# from sklearn.linear_model import LinearRegression

# lin_reg_model = make_pipeline(preprocessing, LinearRegression())
# lin_reg_model.fit(housing_features, housing_labels)

## RMSE VALUE
# housing_predictions = lin_reg_model.predict(housing_features)
# lin_reg_rmse = root_mean_squared_error(housing_labels, housing_predictions)
# print(lin_reg_rmse)

## CROSS VALIDATION
# lin_reg_rmses = -cross_val_score(lin_reg_model, housing_features, housing_labels, scoring="neg_root_mean_squared_error", cv=10)
# print(pd.Series(lin_reg_rmses).describe())

# The model is giving varieties of data with some predictions significantly less (~50%)


### Decision Tree

In [26]:
# from sklearn.tree import DecisionTreeRegressor

# tree_reg_model = make_pipeline(preprocessing, DecisionTreeRegressor(random_state=42))
# tree_reg_model.fit(housing_features, housing_labels)

## RMSE VALUE
# housing_predictions_tree = tree_reg_model.predict(housing_features)
# tree_rmse = root_mean_squared_error(housing_labels, housing_predictions_tree)
# print(tree_rmse)

## Using Cross Validation
# tree_rmses = -cross_val_score(tree_reg_model, housing_features, housing_labels, scoring="neg_root_mean_squared_error", cv = 10)
# print(pd.Series(tree_rmses).describe())


# The model shows no error indicating a very high possibility of overfitting. 


Both the models show significant validation error, decision tree has 67147 rmse score and lin regression has 71266 rmse score. 
Thus, we try to use a new model that would provide better results

### Random Forest

In [27]:
# from sklearn.ensemble import RandomForestRegressor

# forest_reg_model = make_pipeline(preprocessing, RandomForestRegressor(random_state=42))
# forest_reg_model.fit(housing_features, housing_labels)

# housing_predictions_forest = forest_reg_model.predict(housing_features)

# ## RMSE Score
# rf_rmse = root_mean_squared_error(housing_predictions_forest, housing_labels)
# print(rf_rmse) # 17694.738381838666

## CROSS VALIDATION
# rf_rmses = -cross_val_score(forest_reg_model, housing_features, housing_labels, scoring="neg_root_mean_squared_error", cv=10)
# pd.Series(rf_rmses).describe() # mean     66868.027288

# EXTREMELY COMPUTATIONALLY EXPENSIVE TOOK ABOUT 8 MINUTES??? No need to run cross

### SVM

In [28]:
# from sklearn.svm import SVR

# svm_model = make_pipeline(preprocessing, SVR(kernel="linear", C=1.0))
# svm_rmses = -cross_val_score(svm_model, housing_features, housing_labels, scoring="neg_root_mean_squared_error", cv = 10)

# pd.Series(svm_rmses).describe()


### Neural Network

In [29]:
# from sklearn.neural_network import MLPRegressor

# nn_model = make_pipeline(preprocessing, MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42))
# nn_model.fit(housing_features, housing_labels)
# housing_predictions_nn = nn_model.predict(housing_features)

# ## RMSE Score
# nn_rmse = root_mean_squared_error(housing_predictions_nn, housing_labels)
# print(nn_rmse) #52630.13866595486

## CROSS Validation
# nn_rmses = -cross_val_score(nn_model, housing_features, housing_labels, scoring="neg_root_mean_squared_error", cv=3)
# pd.Series(nn_rmses).describe()


### XGBoost

In [30]:
from xgboost import XGBRegressor

xgb_model = make_pipeline(preprocessing, XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42))

xgb_model.fit(housing_features, housing_labels)
housing_predictions_xgb = xgb_model.predict(housing_features)

# ## Calculating RMSE
# xgb_rmse = root_mean_squared_error(housing_predictions_xgb, housing_labels)
# print(xgb_rmse) # 35767.878666788216

## Cross Validation
# xgb_rmses = -cross_val_score(xgb_model, housing_features, housing_labels, scoring="neg_root_mean_squared_error", cv = 10)
# pd.Series(xgb_rmses).describe() # mean     46124.316735


We have try a bunch of models and know which ones provide with the best results. Cross validation is a good way to know when the model is 
over fitting because if the model has very low training error but has a high validation error then it's probably overfitting

After running the cross val for expensive algorithms like SVR-rbf and RF, always comment out or remove them so that they don't run 10 models everytime 
you restart the code

After trying 6 models, the best results have been shown by XGBoost

## Fine-tuning of Model

In [33]:
param_grid = {
    "xgb__max_depth": [4, 6, 8],
    "xgb__learning_rate": [0.05, 0.1, 0.2],
    "xgb__n_estimators": [100, 200]
}

grid_search = GridSearchCV(xgb_model, param_grid, cv=3, scoring="neg_root_mean_squared_error")
grid_search.fit(housing_features, housing_labels)

grid_search.best_params_
cv_res = pd.DataFrame(grid_search.cv_results_)
cv_res.sort_values(by="mean_test_score", ascending=False, inplace=True)
cv_res.head()  # note: the 1st column is the row ID

ValueError: Invalid parameter 'xgb' for estimator Pipeline(steps=[('columntransformer',
                 ColumnTransformer(remainder=Pipeline(steps=[('simpleimputer',
                                                              SimpleImputer(strategy='median')),
                                                             ('standardscaler',
                                                              StandardScaler())]),
                                   transformers=[('bedrooms',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('functiontransformer',
                                                                   FunctionTransformer(feature_names_out=<function ratio_name at 0x000...
                              feature_types=None, feature_weights=None,
                         